# Knee Bone Age — GPU training

Run top to bottom. Set **Runtime → Change runtime type → GPU** first.

## What we are actually fixing

The best local run scored 2.37 y test MAE against a 3.50 y mean-age baseline, so it
learned *something*. Two measurements say the problem is not what it looks like.

**1. It hedges toward the middle.** Predicted-on-true slope was **0.586**: a true 3.2-year-old
came back as 5.9, a true 17.9-year-old as 11.9. Predictions spanned 5.0–15.7 y over a true
3.2–17.9 y range.

**2. Ridge regression on raw voxels beats it.** On the same 240 scans at the same
16×96×96 grid — no convolutions, no pretraining, no augmentation:

| | MAE | slope | corr | within 1 y |
|---|---|---|---|---|
| mean-age baseline | 3.74 y | — | — | — |
| **ridge on raw voxels** | **1.95 y** | **0.686** | 0.836 | 30% |
| 3D ResNet-34 (v3) | 2.37 y | 0.586 | 0.754 | 17% |

A 63M-parameter pretrained network losing to linear regression is not short of data,
and it is not short of resolution.

## Why this notebook no longer chases resolution

An earlier version trained at 32×192×192 on the theory that the growth plate (0.2–3.2 mm)
was sub-voxel at 16×96×96 and therefore invisible. That theory was tested and it is wrong.
The same ridge probe, on identical phantoms at both grids:

| grid | voxels | probe MAE | slope | corr |
|---|---|---|---|---|
| 16×96×96 | 147,457 | 1.453 y | 0.815 | 0.902 |
| 32×192×192 | 1,179,649 | 1.479 y | 0.808 | 0.902 |

Eight times the voxels recovers **nothing**. The physis is a broad plate, so it still fills
voxels laterally even when it is thin. So this notebook trains at **16×96×96** and spends
the saved compute on more scans, more epochs, and an A/B of the output head instead.

**The bar to clear is 1.95 y MAE at slope 0.69**, not the mean-age baseline.

In [ ]:
# 0. Environment, and the knobs worth thinking about.
!nvidia-smi
import torch; print('CUDA available:', torch.cuda.is_available())

GRID = '16 96 96'   # 32 192 192 measurably buys nothing and costs 8x
EPOCHS = 60         # cheap at this grid; local runs only ever got 12-15
TOTAL_SCANS = 600   # local runs used 240
print(f'grid {GRID} | {EPOCHS} epochs | {TOTAL_SCANS} scans')

In [ ]:
# 1. Get the code
!git clone https://github.com/anshppatel4-crypto/knee-bone-age-ai.git
%cd knee-bone-age-ai
!pip install -q monai pydicom scikit-image

In [ ]:
# 2. Generate phantoms. Measured at 8.4 s/scan on one core, so this is CPU-bound and is
#    the longest step. Generation stays at full 32x256x256 fidelity regardless of GRID --
#    preprocessing downsamples from it, so rendering coarsely would throw away detail
#    before the choice of grid is even made.
import os, subprocess, time

cores = os.cpu_count() or 2
per_job = -(-TOTAL_SCANS // cores)
print(f'{cores} cores -> {cores} jobs x {per_job} scans; estimate '
      f'{TOTAL_SCANS * 8.4 / cores / 60:.0f} min')

start = time.time()
jobs = [subprocess.Popen(['python', '-m', 'src.knee_phantom', '--count', str(per_job),
                          '--out', f'data/phantom_gpu_{i:02d}', '--seed', str(31 + i),
                          '--slices', '32', '--resolution', '256'])
        for i in range(cores)]
codes = [job.wait() for job in jobs]
print(f'generation took {(time.time() - start) / 60:.1f} min; exit codes {codes}')
assert not any(codes), 'a generation job failed -- read its traceback above before training'
!ls -d data/phantom_gpu_*/knee_* | wc -l

In [ ]:
# 3. Establish the floor on THIS cohort before training anything.
#    Ridge on raw voxels, no network. Whatever it scores is obtainable from the pixels by
#    the simplest thing that works, so it is the number the ResNet has to beat.
!python src/probe_baseline.py --data 'data/phantom_gpu_*' --input-shape {GRID} --out probe_floor.json

In [ ]:
# 4. Train both output heads: same data, same split, same seed.
#
#    sigmoid(x)*20 bounds the age but scales the gradient by 20*s*(1-s), so the ends of
#    the range learn about half as fast as the middle -- a mechanism for that 0.586
#    slope. --head linear drops the squashing. Which wins is empirical, so run both.
#
#    --workers matters as much as the GPU: at the default 0 the GPU waits on CPU
#    augmentation for every single sample. Drop --batch-size if you hit CUDA OOM.
import os
workers = min(4, max(2, os.cpu_count() or 2))
print('dataloader workers:', workers)

for head in ['sigmoid', 'linear']:
    print(f'\n{"=" * 60}\n  HEAD: {head}\n{"=" * 60}', flush=True)
    !python src/train.py --data 'data/phantom_gpu_*' --arch resnet34 --epochs {EPOCHS} --batch-size 8 --lr 3e-4 --workers {workers} --head {head} --input-shape {GRID} --output final_knee_model_{head}.pth

In [ ]:
# 5. Compare. Slope is the number that matters, not MAE.
import json, numpy as np, sys
sys.path.insert(0, '.')
from torch.utils.data import DataLoader
from src.model import load_checkpoint
from src.predict import predict_scan
from src.dataset import KneeVolumeDataset
from src.train import build_catalog, split_catalog, predict_loader, regression_metrics

test = split_catalog(build_catalog(['data/phantom_gpu_*']), seed=0)[2]
baseline = json.load(open('final_knee_model_sigmoid_metrics.json'))['baseline']['test']['mae']
floor = json.load(open('probe_floor.json'))  # measured on THIS cohort in cell 3
print(f"mean-age baseline {baseline:.3f} y | ridge floor {floor['mae']:.3f} y at slope "
      f"{floor['slope']:.3f} (n={floor['n']}) | local v3 was 2.374 y at slope 0.586\n")

results = {}
for head in ['sigmoid', 'linear']:
    model, meta = load_checkpoint(f'final_knee_model_{head}.pth', 'cuda')
    dataset = KneeVolumeDataset(test, input_shape=tuple(meta['input_shape']), augment=False)
    pred, targ = predict_loader(model, DataLoader(dataset, batch_size=8), 'cuda', tta=True)
    m = regression_metrics(pred, targ)
    results[head] = (m, pred, targ, model, dataset)
    print(f"{head:>8}: MAE {m['mae']:.3f}y | slope {m['slope']:.3f} | corr {m['corr']:.3f} | "
          f"within 1y {m['within_1y']:.0%} | range {m['pred_range'][0]:.1f}-{m['pred_range'][1]:.1f}y")

best = max(results, key=lambda h: results[h][0]['slope'])
m, pred, targ, model, dataset = results[best]
print(f"\nbetter slope: {best}")
# Fold assignment moves the floor by a few hundredths of a year, so allow a tolerance.
beat = m['mae'] < floor['mae'] + 0.05 and m['slope'] > floor['slope'] - 0.02
print(f"{'BEAT' if beat else 'LOST TO'} the ridge floor "
      f"({m['mae']:.3f} vs {floor['mae']:.3f} y, slope {m['slope']:.3f} "
      f"vs {floor['slope']:.3f})")

# Per age band. Local runs were worst at 17+ (4.6 y), where the physis has fused and the
# remaining cue is marrow conversion rather than plate thickness.
bands = np.digitize(targ, [6, 9, 12, 15, 17])
print(f'\nper age band ({best}):')
for band, label in enumerate(['<6', '6-9', '9-12', '12-15', '15-17', '17+']):
    mask = bands == band
    if mask.any():
        print(f'  {label:>6}: n={mask.sum():3d}  MAE {np.abs(pred[mask] - targ[mask]).mean():.3f}y')

# Sex must matter. The phantom defines maturity as age + 1.8 y for girls, so for a fixed
# image, being told "female" should return an age ~1.8 y LOWER than "male". Local runs
# measured 0.02 y, which meant sex was being ignored outright.
deltas = [predict_scan(model, dataset[i]['image'].numpy()[0], 'm', 'cuda')['bone_age']
          - predict_scan(model, dataset[i]['image'].numpy()[0], 'f', 'cuda')['bone_age']
          for i in range(min(15, len(dataset)))]
print(f'\nsex delta (M-F): {np.mean(deltas):+.3f} years (want ~+1.8)')

In [ ]:
# 6. Download both runs
from google.colab import files
for head in ['sigmoid', 'linear']:
    files.download(f'final_knee_model_{head}.pth')
    files.download(f'final_knee_model_{head}_metrics.json')

## Reading the result

Read these in order — each can invalidate the one above it.

1. **Did either head beat the ridge floor?** Printed in cell 5. If neither reaches
   ~1.95 y MAE at slope ~0.69, the network is still losing to linear regression on raw
   pixels, and nothing about the headline MAE is worth celebrating. That is the result
   that tells us where to go next.
2. **Slope, not MAE.** Locally 0.586. A model can improve MAE purely by hedging harder
   toward the mean; slope is what separates that from actually reading anatomy.
3. **Sex delta ≈ +1.8 y.** Near zero means sex is being ignored, which the phantom labels
   definitely encode.
4. **Train loss well below val MAE.** Finally fitting. Locally they were about equal —
   the signature of underfitting.

### If it still loses to the ridge probe

Then the trunk or the schedule is wrong, not the data. In rough order of what I would
try: `--arch resnet18` (63M parameters on a few hundred scans is a lot of model), a
higher `--lr` with more epochs, or dropping `pretrained=True` — MedicalNet was trained on
CT and adult MR, and those filters may simply not transfer to paediatric knee MR.

### The caveat that does not go away

Every number here is accuracy **on synthetic phantoms**, scored against labels the
generator wrote itself. It shows the network can recover maturity from anatomy the
generator drew. It says nothing about error on a real patient. The sex model in
particular is a flat 1.8-year offset, whereas real knee maturation diverges by sex in a
way that varies with age. Validating on real scans is a separate job from this one.